In [2]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

## Define nodes

In [3]:
# Define nodes
station_codes = pd.read_csv('mrt_lrt_stations.csv')
# exclude all the stations that LINE_COLOR is not grey
nodes = station_codes[station_codes['LINE_COLOR'] != 'Grey']['ALPHANUMERIC_CODE'].unique().tolist()
nodes.insert(nodes.index('EW33') + 1, 'CG0') # insert CG0 because tanah merah has no CG code but it is an interchange to go to expo/CG
nodes.insert(nodes.index('CC29') + 1, 'CE0') # insert CE0 because promenade has no CE code but it is an interchange to go to sentosa/CE
# nodes

### Pull only the downtown line, i.e. starting with 'DT'


In [4]:
# nodes is a list.
# print(type(nodes))
dtl_stations = [station for station in nodes if station.startswith('DT')]
dtl_stations

['DT1',
 'DT2',
 'DT3',
 'DT4',
 'DT5',
 'DT6',
 'DT7',
 'DT8',
 'DT9',
 'DT10',
 'DT11',
 'DT12',
 'DT13',
 'DT14',
 'DT15',
 'DT16',
 'DT17',
 'DT18',
 'DT19',
 'DT20',
 'DT21',
 'DT22',
 'DT23',
 'DT24',
 'DT25',
 'DT26',
 'DT27',
 'DT28',
 'DT29',
 'DT30',
 'DT31',
 'DT32',
 'DT33',
 'DT34',
 'DT35']

## Define arcs

In [5]:
# each node is connected to the next node in the same line, and also to the interchanges
# the nodes have values e.g. NS1, NS2, NS3. NS1 has an arc to NS2, NS2 has an arc to NS3 but NS1 has no arc to NS3.
arcs = []
for i in range(len(nodes) - 1):
    if nodes[i][:2] == nodes[i + 1][:2]:
        arcs.append((nodes[i], nodes[i + 1]))
# print(arcs)
interchanges = [('NS1', 'EW24'), ('NS9', 'TE2'), ('CE0', 'CC4'), ('CE0', 'DT16'),
                ('NS17', 'CC15'), ('NS21', 'DT11'), ('NS22', 'TE14'), 
               ('NS24', 'NE6'), ('NS24', 'CC1'), ('NS25', 'EW13'), 
               ('NS26', 'EW14'), ('NS27', 'TE20'), ('NS27', 'CE2'),
               ('CC22', 'EW21'), ('EW16', 'NE3'), ('EW16', 'TE17'), 
               ('EW12', 'DT14'), ('EW8', 'CC9'), ('EW4', 'CG0'), 
               ('EW2', 'DT32'), ('CG1', 'DT35'), ('CC19', 'DT9'),
               ('DT10', 'TE11'), ('NE7', 'DT12'), ('DT15', 'CC4'),
               ('DT16', 'CE1'), ('DT19', 'NE4'), ('DT26', 'CC10'),
               ('CC29', 'NE1'), ('CC17', 'TE9'), ('CC13', 'NE12'),
               ('CC1', 'NE6'), ('CE2', 'TE20'), ('NE3', 'TE17'), 
               ]
for interchange in interchanges:
    arcs.append(interchange)
# since all arcs are bidirectional, need to include the other direction as well.
for arc in arcs.copy():
    arcs.append((arc[1], arc[0]))

print(arcs)

[('NS1', 'NS2'), ('NS2', 'NS3'), ('NS3', 'NS4'), ('NS4', 'NS5'), ('NS5', 'NS7'), ('NS7', 'NS8'), ('NS8', 'NS9'), ('NS9', 'NS10'), ('NS10', 'NS11'), ('NS11', 'NS12'), ('NS12', 'NS13'), ('NS13', 'NS14'), ('NS14', 'NS15'), ('NS15', 'NS16'), ('NS16', 'NS17'), ('NS17', 'NS18'), ('NS18', 'NS19'), ('NS19', 'NS20'), ('NS20', 'NS21'), ('NS21', 'NS22'), ('NS22', 'NS23'), ('NS23', 'NS24'), ('NS24', 'NS25'), ('NS25', 'NS26'), ('NS26', 'NS27'), ('NS27', 'NS28'), ('EW1', 'EW2'), ('EW2', 'EW3'), ('EW3', 'EW4'), ('EW4', 'EW5'), ('EW5', 'EW6'), ('EW6', 'EW7'), ('EW7', 'EW8'), ('EW8', 'EW9'), ('EW9', 'EW10'), ('EW10', 'EW11'), ('EW11', 'EW12'), ('EW12', 'EW13'), ('EW13', 'EW14'), ('EW14', 'EW15'), ('EW15', 'EW16'), ('EW16', 'EW17'), ('EW17', 'EW18'), ('EW18', 'EW19'), ('EW19', 'EW20'), ('EW20', 'EW21'), ('EW21', 'EW22'), ('EW22', 'EW23'), ('EW23', 'EW24'), ('EW24', 'EW25'), ('EW25', 'EW26'), ('EW26', 'EW27'), ('EW27', 'EW28'), ('EW28', 'EW29'), ('EW29', 'EW30'), ('EW30', 'EW31'), ('EW31', 'EW32'), ('EW3

#### Pull the downtown line arcs

In [6]:
dtl_arcs = [arc for arc in arcs if arc[0] in dtl_stations and arc[1] in dtl_stations]
dtl_arcs

[('DT1', 'DT2'),
 ('DT2', 'DT3'),
 ('DT3', 'DT4'),
 ('DT4', 'DT5'),
 ('DT5', 'DT6'),
 ('DT6', 'DT7'),
 ('DT7', 'DT8'),
 ('DT8', 'DT9'),
 ('DT9', 'DT10'),
 ('DT10', 'DT11'),
 ('DT11', 'DT12'),
 ('DT12', 'DT13'),
 ('DT13', 'DT14'),
 ('DT14', 'DT15'),
 ('DT15', 'DT16'),
 ('DT16', 'DT17'),
 ('DT17', 'DT18'),
 ('DT18', 'DT19'),
 ('DT19', 'DT20'),
 ('DT20', 'DT21'),
 ('DT21', 'DT22'),
 ('DT22', 'DT23'),
 ('DT23', 'DT24'),
 ('DT24', 'DT25'),
 ('DT25', 'DT26'),
 ('DT26', 'DT27'),
 ('DT27', 'DT28'),
 ('DT28', 'DT29'),
 ('DT29', 'DT30'),
 ('DT30', 'DT31'),
 ('DT31', 'DT32'),
 ('DT32', 'DT33'),
 ('DT33', 'DT34'),
 ('DT34', 'DT35'),
 ('DT2', 'DT1'),
 ('DT3', 'DT2'),
 ('DT4', 'DT3'),
 ('DT5', 'DT4'),
 ('DT6', 'DT5'),
 ('DT7', 'DT6'),
 ('DT8', 'DT7'),
 ('DT9', 'DT8'),
 ('DT10', 'DT9'),
 ('DT11', 'DT10'),
 ('DT12', 'DT11'),
 ('DT13', 'DT12'),
 ('DT14', 'DT13'),
 ('DT15', 'DT14'),
 ('DT16', 'DT15'),
 ('DT17', 'DT16'),
 ('DT18', 'DT17'),
 ('DT19', 'DT18'),
 ('DT20', 'DT19'),
 ('DT21', 'DT20'),
 ('DT22'

## Define costs
* Taken from the dataset that has the timings between stations, including transfer timings
* Link: https://docs.google.com/spreadsheets/d/13CTjSiDdrjvIJvvCso10TWSP5Bf1kZrR77W638BVFkI/edit?gid=987939193#gid=987939193

In [7]:
station_timings_cost = pd.read_csv('station_costs_no_interchanges.csv')
# Map each station to cost given the codes
costs = {}
# print(station_timings_cost.iloc[2]['station_code'])
for index, row in station_timings_cost.iterrows():
    station_code = row['station_code']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    prev_station_code = station_timings_cost.iloc[index-1]['station_code'] if index >= 0 else None
    # convert travel_time from 0:04:25 to number of seconds, which is 4*60 + 25 = 265 seconds
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    if prev_station_code and station_code[:2] == prev_station_code[:2] and cost_in_seconds != 0: # if the station is on the same line as the previous station, then the cost is the travel time between them
        costs[(prev_station_code, station_code)] = cost_in_seconds
        costs[(station_code, prev_station_code)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions
# print(costs)
# need to manually add the costs for transfer at interchanges
station_timings_cost_interchanges = pd.read_csv('station_costs_interchanges.csv')
station_timings_cost_interchanges = station_timings_cost_interchanges.dropna()
for index, row in station_timings_cost_interchanges.iterrows():
    station_code_1 = row['station_code_1']
    station_code_2 = row['station_code_2']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    costs[(station_code_1, station_code_2)] = cost_in_seconds
    costs[(station_code_2, station_code_1)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions

print(costs)

{('TE1', 'TE2'): 116, ('TE2', 'TE1'): 116, ('TE2', 'TE3'): 152, ('TE3', 'TE2'): 152, ('TE3', 'TE4'): 265, ('TE4', 'TE3'): 265, ('TE4', 'TE5'): 183, ('TE5', 'TE4'): 183, ('TE5', 'TE6'): 139, ('TE6', 'TE5'): 139, ('TE6', 'TE7'): 129, ('TE7', 'TE6'): 129, ('TE7', 'TE8'): 164, ('TE8', 'TE7'): 164, ('TE8', 'TE9'): 200, ('TE9', 'TE8'): 200, ('TE9', 'TE11'): 217, ('TE11', 'TE9'): 217, ('TE11', 'TE12'): 190, ('TE12', 'TE11'): 190, ('TE12', 'TE13'): 122, ('TE13', 'TE12'): 122, ('TE13', 'TE14'): 128, ('TE14', 'TE13'): 128, ('TE14', 'TE15'): 142, ('TE15', 'TE14'): 142, ('TE15', 'TE16'): 90, ('TE16', 'TE15'): 90, ('TE16', 'TE17'): 112, ('TE17', 'TE16'): 112, ('TE17', 'TE18'): 102, ('TE18', 'TE17'): 102, ('TE18', 'TE19'): 119, ('TE19', 'TE18'): 119, ('TE19', 'TE20'): 109, ('TE20', 'TE19'): 109, ('TE20', 'TE22'): 164, ('TE22', 'TE20'): 164, ('TE22', 'TE23'): 198, ('TE23', 'TE22'): 198, ('TE23', 'TE24'): 125, ('TE24', 'TE23'): 125, ('TE24', 'TE25'): 132, ('TE25', 'TE24'): 132, ('TE25', 'TE26'): 113, 

#### Pull the downtown line costs

In [8]:
# a single entry in the cost dictionary is ('TE1', 'TE2'): 116
# take a subset of those costs whose keys are in dtl_arcs, i.e. both stations are in dtl_stations
dtl_costs = {arc: cost for arc, cost in costs.items() if arc in dtl_arcs}
# dtl_costs
print(len(dtl_costs))

68


Passes sanity check. we have 34*2 =68 edges total

#### Create the base model with decision variables

In [9]:
print(len(dtl_stations))

35


In [32]:
TRAIN_COUNT = 100 # can be modified later
m = gp.Model('minimize total passenger travel time + waiting penalty + operating costs')
STATION_COUNT = len(dtl_stations)
# create a TRAIN_COUNT by STATION_COUNT matrix of binary decision variables, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
x = [[0 for j in range(STATION_COUNT)] for t in range(TRAIN_COUNT)]
# Create decision variables for each station and each train, where x[t][j] = 1 if train t stops at station j, and 0 otherwise
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        x[t][j] = m.addVar(vtype=GRB.BINARY, name=f'x_{t}_{j}')

y = [0 for t in range(TRAIN_COUNT)]       
# y[t] is binary, 1 if train t is express, 0 if train t is regular, local train
for t in range(TRAIN_COUNT):
    y[t] = m.addVar(vtype=GRB.BINARY, name=f'y_{t}')
z = [0 for j in range(STATION_COUNT)]
# z[j] is binary, 1 if station j is designated as express station, 0 otherwise
for j in range(STATION_COUNT):
    z[j] = m.addVar(vtype=GRB.BINARY, name=f'z_{j}')

u = [0 for j in range(STATION_COUNT)]
# u[j] >= 0 is the service shortfall at station j
# if u[j] = 0, station j gets enough stopping trains
# if u[j] > 0, station j is under-served and incurs a penalty
for j in range(STATION_COUNT):
    u[j] = m.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f'u_{j}')

print(len(x[0]))

35


#### Constraint 1: terminal stations must always be served and cannot be skipped.

In [11]:
# station DT1 and DT35 must be served by all trains, express or regular, and cannot be skipped.

for t in range(TRAIN_COUNT):
    m.addConstr(x[t][0] == 1)
    m.addConstr(x[t][STATION_COUNT - 1] == 1)

#### Constraint 2: If a train is local, it MUST stop at every station.

In [12]:
# add constraint that says if a train is local, it MUST stop at every station, i.e. xtj >= 1 - yt for all t, for all j

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] >= 1 - y[t]) 

#### Constraint 3: All stations that have an interchange and is located around the CBD must be served

In [27]:
M = set()
for interchange in interchanges:
    if interchange[0] in dtl_stations:
        M.add(interchange[0])
    elif interchange[1] in dtl_stations:
        M.add(interchange[1])

station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}
for t in range(TRAIN_COUNT):
    for station in M:
        m.addConstr(x[t][station_to_idx[station]] == 1, name=f"mandatory_stop_{t}_{station}")

#### Constraint 4: Limit the number of express trains possible in a given set of trains

In [28]:
# Let K be the max number of express trains, where K < TRAIN_COUNT. TODO.
K = 20

m.addConstr(gp.quicksum(y[j] for j in range(TRAIN_COUNT)) <= K)

<gurobi.Constr *Awaiting Model Update*>

#### Constraint 5: Express trains can only stop at designated express stations, i.e. if train t is express it can ONLY stop where the station is express, else if local train, no restriction

In [15]:
# xtj ≤ zj + (1 – yt) ∀ t ∈ T, j ∈ S
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] <= z[j] + (1 - y[t]))

#### Constraint 6: Limit how many stations an express train can skip, if not it may skip the whole thing

In [29]:
# ∑(1-xtj) <= Lyt , ∀ t ∈ T. L is max skipped stations
# check this later. TODO.    

L = 10

for t in range(TRAIN_COUNT):
    m.addConstr(
        gp.quicksum(1 - x[t][j] for j in range(1, STATION_COUNT - 1)) <= L * y[t],
        name=f"limit_skips_{t}"
    )

#### Constraint 7: Service coverage, i.e. station should ideally receive at least xxxx amount of stopping trains. If not, the objective will penalize per-unit. Let xxxx for each station be Hj, the ideal number of trains to stop at station j. Hj should be a set of numbers

In [30]:
H = [1 for _ in range(STATION_COUNT)]  # Example values, replace with actual ideal numbers. TODO

for j in range(STATION_COUNT):
    m.addConstr(gp.quicksum(x[t][j] for t in range(TRAIN_COUNT)) + u[j] >= H[j]) 
    # if the number of trains stopping at station j is less than the ideal number H[j], 
    # then uj will be positive and will pay a penalty in the objective function. if the number of trains stopping at station j is greater than or equal to H[j], then uj will be 0 and there will be no penalty in the objective function.

#### Constraint 8: Let ft is the additional operating cost if train t is express. Budget cap, e.g. for extra signage, passenger info, timetable redesign, operational complexity

In [18]:
# let F[t] be additional operating cost if train t is express.
# we have a budget cap, e.g. for extra signage, timetable etc. TODO.

F = [100 for _ in range(TRAIN_COUNT)]  # Example values, replace with actual costs
BUDGET_CONSTRAINT = 5000 # Example value, replace with actual budget constraint
m.addConstr(gp.quicksum(F[t] * y[t] for t in range(TRAIN_COUNT)) <= BUDGET_CONSTRAINT)


<gurobi.Constr *Awaiting Model Update*>

#### Constraint 9: All express trains must follow the same skip-stop route. E.g. if we have ABCDE as train stations, and the express station are ACE, then all express trains have to follow ACE.

In [33]:
# All express trains must follow the same skip-stop route

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        # If train t is express, it must stop if station j is an express station
        m.addConstr(x[t][j] >= z[j] - (1 - y[t]), name=f"express_lower_{t}_{j}")

#### Constraint 10: Force terminal and mandatory stations (set M) to be in the express set. 

In [34]:
# Force terminal stations to be in the express set
m.addConstr(z[0] == 1, name="terminal_start_in_express_set")
m.addConstr(z[STATION_COUNT - 1] == 1, name="terminal_end_in_express_set")

# Force mandatory stations to be in the express set
station_to_idx = {f"DT{i}": i - 1 for i in range(1, STATION_COUNT + 1)}

for station in M:
    m.addConstr(z[station_to_idx[station]] == 1, name=f"mandatory_express_{station}")

#### Constraint 11 [OPTIONAL]: Force that express stations is a strict subset of all the total stations available. This is because we don't want the solution where all stations become express stations, otherwise the optimizer is still allowed to choose all stations as express stations, making the express identical to local.

In [36]:
# OPTIONAL. Uncomment if we want to limit the number of express stations, which will in turn limit the number of possible skip-stop patterns for the express trains. TODO.
# m.addConstr(gp.quicksum(z[j] for j in range(STATION_COUNT)) <= STATION_COUNT - 1, name="limit_express_stations")

#### Set objective and optimize

First term: "If train t stops at station j, the system pays a passenger travel-time cost equal to fixed stop time k times passenger impact qj." And xtj is the binary decision variable described above.
* k: Travel-time coefficient, which is a fixed constant time penalty per stop, as we know, when a train stops at a station, there is extra travel time due to deceleration, acceleration and dwell time. k basically sums of all that costs.
* qj: passenger's weight associated with stopping at station j. Represents how many passengers are affected when a train stops at station j. A stop at a busier part of the line may delay more onboard passengers, so qj may be larger there

Second term: if a station gets fewer stopping trains than desired, we pay a penalty. Prevents the model from making it skip everything and harm all the passengers at the skipped stations. pj is the waiting penalty per unit of under-service at station j. Uj is the amount of service shortfall at station j
* pj is the penalty per unit of under-service at station j, like per-unit underproduction costs of baking a cake. Larger pj nearer to the CBD as that means the station is important, under-serving it is bad. Smaller pj means model is more willing to reduce service there.
* uj is described above, as the amount of units that is underproduced. </br>

Third term: ft is the additional operating cost if train t is express, yt is defined above.

Running express trains can create extra cost due to scheduling complexity, operations planning, extra signage, platform information, passenger confusion. Basically non-quantifiable cost

In [39]:
# Example values, replace with actual penalties. This means every unit of under-service costs 10
# TODO
p = [10 for _ in range(STATION_COUNT)] 

# Example values, replace with actual passenger impact weights. Represents how many passengers are affected when a train stops at station j
# TODO
q = [1 for _ in range(STATION_COUNT)]

# Example value, replace with actual travel-time coefficient. This means every stop costs 1
# TODO
k = 1 
travel_time_cost = k * gp.quicksum(
    q[j] * x[t][j]
    for t in range(TRAIN_COUNT)
    for j in range(STATION_COUNT)
)

underservice_cost = gp.quicksum(
    p[j] * u[j]
    for j in range(STATION_COUNT)
)

express_cost = gp.quicksum(
    F[t] * y[t]
    for t in range(TRAIN_COUNT)
)

m.setObjective(
    travel_time_cost + underservice_cost + express_cost,
    GRB.MINIMIZE
)

m.optimize()


if m.status == GRB.OPTIMAL:
    print("Optimal objective value:", m.objVal)
else:
    print("No optimal solution found. Status code:", m.status)
# # Print solution (change later, may be wrong)
# print("Optimal objective value:", m.objVal)
# print("Train stop patterns:")
# for t in range(TRAIN_COUNT):
#     stops = [j for j in range(STATION_COUNT) if x[t][j].X > 0.5]
#     print(f"Train {t}: {stops}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 7500F 6-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 3514 rows, 3670 columns and 10548 nonzeros (Min)
Model fingerprint: 0xb3f52cf3
Model has 3635 linear objective coefficients
Variable types: 35 continuous, 3635 integer (3635 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+01]

MIP start from previous solve produced solution with objective 0 (0.01s)
Loaded MIP start from previous solve with objective 0


Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, ga

## Model ends here for now. below information is not included, may or may not be required.

### Placeholder

## Capacities for each arc
* Should be the same for every connected line, because 1 train will traverse the entire line of the same color.
* if the arc is describing a tranfer arc, i.e. via manual walking e.g. jurong east green to red line, we can assume the capacities to be infinite (capacities here in this case refer to the number of people that can transfer from 1 line to another, which is technically much larger when transiting between lines than the fixed capacity of a single train transporting passengers for a single line)
* referencing this article: https://medium.com/@simpletan/deep-dive-of-singapores-rail-network-passenger-loads-e331d3b9626b

In [20]:
capacities = {}

# Supply and demand at each node
* Distance-defined. To be simulated based on distance from CBD
* Considering only evening peak hour, so supply (i.e. people going into the MRT) is higher as we get closer to the various CBD locations.

In [21]:
supply = {
    1: GRB.INFINITY,
    2: 0,
    3: 0,
    4: 0
}
# just an example of supply. change later

# Constraints

In [ ]:
# flow = m.addVars(arcs,ub=[capacities[arc] for arc in arcs])
# selling_amount = m.addVar(name="selling_amount")

# # Flow conservation constraints at nodes 
# for i in supply:
#     if (i == 1):
#         m.addConstr(flow.sum(i, '*') == selling_amount)
#     elif (i == 5):
#         m.addConstr(flow.sum('*', i) == selling_amount)
#     else:
#         m.addConstr(flow.sum(i, '*') - flow.sum('*', i) == 0)
